In [ ]:
import argparse
import torch
import numpy as np
import random
from peft import (
    LoraConfig,
    get_peft_model
)
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import os
import sys
import json
import transformers
import warnings
from datasets import load_dataset
from predict_module import sft_dataloader

from utils.prompts import PREDICT_INSTRUCTION
from utils.fewshots import PREDICT_EXAMPLES

# Thiết lập seed
fix_seed = 100
random.seed(fix_seed)
torch.manual_seed(fix_seed)
np.random.seed(fix_seed)

# Cấu hình tham số cho huấn luyện
args = argparse.Namespace(
    
    wandb=False,  # Tắt logging với Weights & Biases
    data_path="./data/DeepSeekLLM_top1_stock_technical_indicator_merge_sample.json",  # Đường dẫn file dữ liệu
    output_path="./saved_models/lora-DeepSeek-R1-Distill-Qwen",  # Thư mục lưu mô hình LoRA
    model_path="deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B",  # Mô hình DeepSeek
    eval_steps=200,  # Số bước đánh giá
    save_steps=200,  # Số bước lưu checkpoint
    resume_from_supervised_checkpoint=None,  # Không resume từ checkpoint
    ignore_data_skip="False",  # Không bỏ qua dữ liệu khi resume
    num_reflect_trials=2,  # Số lần thử phản ánh
    datasets_dir="./datasets/",  # Thư mục datasets
    local_rank=0,  # Rank cục bộ cho DDP
    resume_from_reward_checkpoint=False,  # Không resume từ reward checkpoint
    deepspeed=None,  # Không dùng DeepSpeed
    per_device_train_batch_size=4,  # Batch size huấn luyện trên mỗi GPU
    per_device_eval_batch_size=4,  # Batch size đánh giá trên mỗi GPU
    reward_gradient_accumulation_steps=8,  # Số bước tích lũy gradient cho reward
    reward_learning_rate=3e-5,  # Learning rate cho reward
    weight_decay=0.001,  # Trọng số giảm dần
    reward_base_model="deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B",  # Mô hình reward
    bf16=False,  # Sử dụng fp16 thay vì bf16
    num_train_epochs=2,  # Số epoch huấn luyện
    train_subset=100000,  # Số mẫu huấn luyện
    eval_subset=50000,  # Số mẫu đánh giá
    gradient_checkpointing=True,  # Bật gradient checkpointing để tiết kiệm VRAM
    optim="adamw_torch",  # Optimizer AdamW từ PyTorch
    lr_scheduler_type="cosine",  # Lịch trình learning rate kiểu cosine
    reward_adapter="./saved_models/reward_model_deepseek-r1-distill-qwen",  # Adapter reward
    rl_base_model="./saved_models/lora-DeepSeek-R1-Distill-Qwen-adapter-merged",  # Mô hình RL
    tokenizer_name="deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B",  # Tokenizer
    reward_model_name="./saved_models/reward_model_deepseek-r1-distill-qwen-adapter-merged",  # Mô hình reward merged
    log_with=None,  # Không dùng logging cụ thể
    rl_learning_rate=2e-5,  # Learning rate cho RL
    output_max_length=256,  # Độ dài đầu ra tối đa
    mini_batch_size=4,  # Kích thước mini-batch
    batch_size=128,  # Kích thước batch tổng
    ppo_epochs=4,  # Số epoch cho PPO
    rl_gradient_accumulation_steps=32,  # Số bước tích lũy gradient cho RL
    adafactor=False,  # Không dùng Adafactor
    early_stopping=True,  # Bật early stopping
    target_kl=0.1,  # KL target cho RL
    reward_baseline=0,  # Baseline cho reward
    batched_gen=True,  # Tạo batch
    save_freq=100,  # Tần suất lưu
    output_dir="./saved_models/tuning_deepseek_r1_distill_qwen_checkpoints/",  # Thư mục lưu checkpoint
    seed=0,  # Seed cho RL
    num_shots=4,  # Số shots cho few-shot
    save_dir="results/"  # Thư mục lưu kết quả
)


print("Args in experiment:")
print(args)

args.data_path


In [1]:
import pandas as pd
import re
# Đọc tệp CSV Data\summarized\OpenAILLM_top1_stock_data_test.csv

path_OpenAILLM_top1_stock_train = "../Data/summarized/OpenAILLM_top1_stock_data_test.csv"
# Đọc file CSV vào DataFrame
df_loaded = pd.read_csv(path_OpenAILLM_top1_stock_train)

# Hiển thị nội dung DataFrame
print("Nội dung tệp CSV:")

df_loaded

from explain_module.util import summarize_trial, remove_reflections, save_results#, save_agents
from explain_module.agents import PredictReflectAgent
from utils.llm import OpenAILLM, DeepSeekLLM #, FastChatLLM
import os, json
agent_cls = PredictReflectAgent

MAIN_LLM = OpenAILLM()

def split_completion(completion_text):
    completion_text = completion_text.replace("*", "")
    if "Price Movement:" in completion_text:
        completion_text = completion_text.split("Price Movement:", -1)[-1]
        
    if "Explanation:" in completion_text:
        parts = completion_text.split("Explanation:", -1)


        # Dò tìm nhãn Positive/Negative trong cả phần text
        match = re.search(r'(Positive|Negative|positive|negative)', parts[0])
        if match:
            target = match.group(1).capitalize()
        else:
            target = 'Mixed'

        # Làm sạch phần giải thích
        explanation_raw = parts[1].strip()
        explain = "Explanation: " + re.split(r'\n|END OF EXAMPLES', explanation_raw)[0].strip()
    else:
        target = 'Mixed'
        explain = ""
        
    return target, explain


agents = [agent_cls(row['ticker'], row['summary'], row['target'], row['technical_indicator'], predict_llm = MAIN_LLM, reflect_llm= MAIN_LLM) for _, row in df_loaded.iterrows()]
print("Loaded Train Agents.")
agents
i = 1
# Danh sách kết quả
results = []

for agent in agents:
    agent.run()
    target = agent.get_target()


    prompt = agent._build_agent_prompt()
    response = agent.scratchpad.split('Price Movement: ')[-1]

    predict, explain = split_completion(response)

    #
    results.append({
        'prompt': prompt,
        'response': response,
        'predict': predict,
        'explain': explain,
        'target': target
    })
    #
    # if i==2:
    #     break
    print(f"Đã xử lý agent thứ {i}/{len(agents)}")
    i=i+1



# Tạo DataFrame kết quả
df_result = pd.DataFrame(results)

# Lưu nếu cần
df_result.to_csv("ChatGPT_results_predict_explain_technical_indicator_3.csv", index=False)
    
correct, incorrect = summarize_trial(agents)
print(f'Finished Trial 0, Correct: {len(correct)}, Incorrect: {len(incorrect)}')



Nội dung tệp CSV:
Loaded Train Agents.
Positive

Explanation: The stock split news confirming a 20-to-1 split for GOOG on July 15th has generated excitement and anticipation among investors, leading to a surge in the stock price. The high price target from analysts, coupled with discussions around potential future stock splits for Google, has created positive sentiment towards the stock. Additionally, the recent deal between Alphabet's Waymo and Uber Freight for self-driving trucks indicates growth potential and innovation within the company. Despite the technical indicators showing a slightly negative MACD and RSI below overbought levels, the overall optimism surrounding GOOG's future prospects and market performance is likely to drive the stock price higher in the near term.



Đã xử lý agent thứ 1/168
Negative

Explanation: The combination of negative stock returns for Google in 2022, along with other tech giants like AAPL, MSFT, TSLA, and AMZN, created a bearish sentiment in the ma

In [2]:
df = pd.read_csv("ChatGPT_results_predict_explain_technical_indicator_3.csv")
df

# Dọn dẹp bộ nhớ GPU Predict_with_tweets_Tech_Indi\ChatGPT_results_predict_explain_technical_indicator.csv
from sklearn.metrics import accuracy_score, matthews_corrcoef

# Giả sử df là DataFrame của bạn
y_pred = df["predict"]
y_true = df["target"]

# Tính accuracy
acc = accuracy_score(y_true, y_pred)

# Tính MCC
mcc = matthews_corrcoef(y_true, y_pred)

print(f"Accuracy: {acc:.4f}")
print(f"MCC: {mcc:.4f}")

Accuracy: 0.5298
MCC: 0.0789


## Explain 

In [ ]:
args
import pandas as pd
# Đọc tệp CSV

path_OpenAILLM_top1_stock_train = "../Data/summarized/OpenAILLM_top1_stock_data_train_sample.csv"
# Đọc file CSV vào DataFrame
df_loaded = pd.read_csv(path_OpenAILLM_top1_stock_train)

# Hiển thị nội dung DataFrame
print("Nội dung tệp CSV:")

df_loaded

from explain_module.util import summarize_trial, remove_reflections, save_results#, save_agents
from explain_module.agents import PredictReflectAgent
from utils.llm import OpenAILLM, DeepSeekLLM #, FastChatLLM
import os, json
agent_cls = PredictReflectAgent

MAIN_LLM = OpenAILLM()

agents = [agent_cls(row['ticker'], row['summary'], row['target'], row['technical_indicator'], predict_llm = MAIN_LLM, reflect_llm= MAIN_LLM) for _, row in df_loaded.iterrows()]
print("Loaded Train Agents.")
agents
i = 1
for agent in agents:
    agent.run()

    if agent.is_correct():
        prompt = agent._build_agent_prompt()
        response = agent.scratchpad.split('Price Movement: ')[-1]
        sample = {"instruction": prompt, "input": "", "output": response}
        with open(args.data_path, 'a') as f:
            f.write(json.dumps(sample) + "\n")
    print(f"Đã xử lý agent thứ {i}/{len(agents)}")
    i=i+1
    
correct, incorrect = summarize_trial(agents)
print(f'Finished Trial 0, Correct: {len(correct)}, Incorrect: {len(incorrect)}')



## Self-Reflection

In [ ]:
# # Train supervised policy
# supervised_finetune(self.args)
# merge_peft_adapter(model_name=self.args.output_path, output_name=self.args.rl_base_model)
print('===================================================')
print('Collect comparison data')

# Collect comparison data
comparison_data = []
for trial in range(args.num_reflect_trials):
    for idx, agent in enumerate([a for a in agents if not a.is_correct()]):
        prev_response = agent.scratchpad.split('Price Movement: ')[-1]
        agent.run()
        if agent.is_correct():
            print(agent._build_agent_prompt(), "\n\n\n")
            prompt = remove_reflections(agent._build_agent_prompt())
            response = agent.scratchpad.split('Price Movement: ')[-1]
            sample = {"user_input": prompt, "completion_a": prev_response, "completion_b": response}
            comparison_data.append(sample)
    correct, incorrect = summarize_trial(agents)
    print(f'Finished Trial {trial+1}, Correct: {len(correct)}, Incorrect: {len(incorrect)}')
os.makedirs(args.datasets_dir, exist_ok=True)
comparison_data_path = os.path.join(args.datasets_dir, "DeepSeekLLM_top1_stock_technical_indicator_comparison_data.json")
if comparison_data:
    with open(comparison_data_path, 'w') as f:
        f.write(json.dumps(comparison_data))